# Ensemble Federated Learning - Google Colab Testing

This notebook tests the `run_ensemble.py` script in Google Colab.

**Steps:**
1. Setup environment and install dependencies
2. Clone/upload your repository
3. Run a quick test with minimal parameters
4. View results

## 1. Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Install Dependencies

Install required packages (most are already in Colab, but let's ensure we have everything).

In [ ]:
!pip install -q torch torchvision scikit-learn

## 3. Clone Repository

**Option A: Clone from GitHub** (if you have pushed to GitHub):

In [ ]:
# Uncomment and modify with your GitHub repository URL
# !git clone https://github.com/YOUR_USERNAME/EnsembleFedLearning.git
# %cd EnsembleFedLearning

**Option B: Upload from Google Drive:**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Modify this path to where your project is stored in Google Drive
# %cd /content/drive/MyDrive/EnsembleFedLearning

**Option C: Upload files directly:**

In [ ]:
# Upload the entire project as a zip file
from google.colab import files
import zipfile

# uploaded = files.upload()  # Upload your zipped project
# !unzip -q EnsembleFedLearning.zip
# %cd EnsembleFedLearning

## 4. Verify Project Structure

In [ ]:
import os
print("Current directory:", os.getcwd())
print("\nProject structure:")
!ls -la

print("\nChecking key files:")
required_files = ['run_ensemble.py', 'config.json', 'training/ensemble_fl.py', 'data/loader.py']
for file in required_files:
    exists = os.path.exists(file)
    print(f"{'✓' if exists else '✗'} {file}")

## 5. Create Quick Test Configuration

Create a minimal config for fast testing (few clients, few rounds).

In [ ]:
import json

test_config = {
    "dataset": "cifar10",
    "data_dir": "./data",
    "alpha": 0.5,
    
    "num_clients": 10,  # Very small for quick testing
    "num_clusters": 2,
    "batch_size": 64,
    "lr": 0.01,
    
    "model_name": "resnet18",
    "pretrained": False,
    
    "warmup_rounds": 1,
    "warmup_local_epochs": 1,  # Reduced for speed
    "use_fedavg_warmup": True,
    "use_weight_diff": False,
    
    "clustering_method": "kmeans",
    "num_feature_samples": 50,
    
    "ensemble_rounds": 5,  # Very few rounds for testing
    "ensemble_local_epochs": 1,
    "client_fraction": 0.5,
    
    "device": "cuda",
    "seed": 42,
    
    "output_dir": "./results_colab_test",
    "save_warmup_model": False,
    "save_final_model": True
}

# Save to file
with open('config_colab_test.json', 'w') as f:
    json.dump(test_config, f, indent=2)

print("Created test configuration:")
print(json.dumps(test_config, indent=2))

## 6. Run Quick Test

Run the ensemble training with minimal parameters to verify everything works.

In [ ]:
!python run_ensemble.py --config config_colab_test.json

## 7. Check Results

In [ ]:
# List output files
print("Results directory contents:")
!ls -lh results_colab_test/

print("\nModels directory:")
!ls -lh results_colab_test/models/

## 8. Load and Visualize Training History

In [ ]:
import json
import matplotlib.pyplot as plt

# Load training history
with open('results_colab_test/training_history.json', 'r') as f:
    history = json.load(f)

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history['rounds'], history['accuracies'], marker='o', linewidth=2)
ax1.set_xlabel('Round', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12)
ax1.set_title('Test Accuracy over Rounds', fontsize=14)
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history['rounds'], history['losses'], marker='o', color='red', linewidth=2)
ax2.set_xlabel('Round', fontsize=12)
ax2.set_ylabel('Test Loss', fontsize=12)
ax2.set_title('Test Loss over Rounds', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nTraining Summary:")
print(f"Best Accuracy: {max(history['accuracies']):.4f} at round {history['rounds'][history['accuracies'].index(max(history['accuracies']))]}")
print(f"Final Accuracy: {history['accuracies'][-1]:.4f}")
print(f"Best Loss: {min(history['losses']):.4f}")
print(f"Final Loss: {history['losses'][-1]:.4f}")

## 9. View Cluster Assignments

In [ ]:
import json
from collections import Counter

# Load cluster assignments
with open('results_colab_test/models/client_clusters.json', 'r') as f:
    client_clusters = json.load(f)

# Convert string keys to integers
cluster_assignments = [int(client_clusters[str(i)]) for i in range(len(client_clusters))]

# Count clients per cluster
cluster_counts = Counter(cluster_assignments)

print("Cluster Distribution:")
for cluster_id in sorted(cluster_counts.keys()):
    print(f"  Cluster {cluster_id}: {cluster_counts[cluster_id]} clients")

# Visualize distribution
plt.figure(figsize=(8, 5))
plt.bar(cluster_counts.keys(), cluster_counts.values())
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Number of Clients', fontsize=12)
plt.title('Client Distribution Across Clusters', fontsize=14)
plt.grid(axis='y', alpha=0.3)
plt.show()

## 10. Run Full Training (Optional)

Once the quick test works, you can run a full training session with more clients and rounds.

In [ ]:
# Run with default config (100 clients, 50 rounds)
# This will take much longer!
# !python run_ensemble.py --config config.json

## 11. Download Results (Optional)

Download the trained models and results to your local machine.

In [ ]:
# Zip the results
!zip -r results_colab_test.zip results_colab_test/

# Download
from google.colab import files
# files.download('results_colab_test.zip')

## Troubleshooting

### Out of Memory
- Reduce `num_clients`: `--num_clients 10`
- Reduce `batch_size`: `--batch_size 32`
- Reduce `num_clusters`: `--num_clusters 2`

### Runtime Disconnection
- Use fewer rounds for initial testing
- Save checkpoints more frequently
- Consider upgrading to Colab Pro for longer runtime

### Import Errors
- Check that you're in the correct directory
- Verify all files are uploaded correctly
- Make sure dependencies are installed